In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [2]:
df =pd.read_excel('../datasets/Superstores Sales/superstore_sales.xlsx')

In [3]:
print(df[['Order Date', 'Ship Date']].sample(10, random_state=42))

               Order Date            Ship Date
532   2018-07-09 00:00:00  2018-11-09 00:00:00
872   2015-10-12 00:00:00           15/12/2015
1149  2016-04-04 00:00:00  2016-04-04 00:00:00
2287  2018-01-06 00:00:00  2018-05-06 00:00:00
4038           29/12/2015  2016-02-01 00:00:00
1726           19/12/2016           20/12/2016
4989           31/08/2018  2018-04-09 00:00:00
4228           24/03/2017           26/03/2017
6664           26/10/2018  2018-01-11 00:00:00
7598           17/09/2015           22/09/2015


In [4]:
# clean up any accidental leading/trailing whitespace in the date columns
df['Order Date'] = df['Order Date'].astype(str).str.strip()
df['Ship Date'] = df['Ship Date'].astype(str).str.strip()

In [5]:
# df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d/%m/%Y', errors='coerce')
# df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d/%m/%Y', errors='coerce')

df['Order Date'] = pd.to_datetime(df['Order Date'], format='mixed', errors='coerce')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='mixed', errors='coerce')

In [6]:
print(df[['Order Date', 'Ship Date']].dtypes)

Order Date    datetime64[ns]
Ship Date     datetime64[ns]
dtype: object


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9800 non-null   int64         
 1   Order ID       9800 non-null   object        
 2   Order Date     9800 non-null   datetime64[ns]
 3   Ship Date      9800 non-null   datetime64[ns]
 4   Ship Mode      9800 non-null   object        
 5   Customer ID    9800 non-null   object        
 6   Customer Name  9800 non-null   object        
 7   Segment        9800 non-null   object        
 8   Country        9800 non-null   object        
 9   City           9800 non-null   object        
 10  State          9800 non-null   object        
 11  Postal Code    9789 non-null   float64       
 12  Region         9800 non-null   object        
 13  Product ID     9800 non-null   object        
 14  Category       9800 non-null   object        
 15  Sub-Category   9800 n

In [8]:
total_duplicates = df.duplicated().sum()
print(f'Total duplicate rows: {total_duplicates}')

Total duplicate rows: 0


In [9]:
subset_duplicates = df.duplicated(subset=['Order ID', 'Product ID']).sum()
print(f'Duplicate rows based on Order ID and Product ID: {subset_duplicates}')

Duplicate rows based on Order ID and Product ID: 8


In [10]:
duplicated_mask = df.duplicated(subset=['Order ID', 'Product ID'], keep=False)
duplicated_rows = df[duplicated_mask].sort_values(by=['Order ID', 'Product ID'])
print(duplicated_rows[['Order ID', 'Product ID']].head(20))

            Order ID       Product ID
6498  CA-2016-103135  OFF-BI-10000069
6500  CA-2016-103135  OFF-BI-10000069
350   CA-2017-129714  OFF-PA-10001970
352   CA-2017-129714  OFF-PA-10001970
1300  CA-2017-137043  FUR-FU-10003664
1301  CA-2017-137043  FUR-FU-10003664
9168  CA-2017-140571  OFF-PA-10001954
9169  CA-2017-140571  OFF-PA-10001954
7881  CA-2018-118017  TEC-AC-10002006
7882  CA-2018-118017  TEC-AC-10002006
3183  CA-2018-152912  OFF-ST-10003208
3184  CA-2018-152912  OFF-ST-10003208
3405  US-2015-150119  FUR-CH-10002965
3406  US-2015-150119  FUR-CH-10002965
430   US-2017-123750  TEC-AC-10004659
431   US-2017-123750  TEC-AC-10004659


In [11]:
display(duplicated_rows[['Order ID', 'Product ID', 'Order Date', 'Ship Date', 'Product Name', 'Ship Mode', 'Sales']])

,Order ID,Product ID,Order Date,Ship Date,Product Name,Ship Mode,Sales
6498,CA-2016-103135,OFF-BI-10000069,2016-07-24,2016-07-28,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",Standard Class,135.090
6500,CA-2016-103135,OFF-BI-10000069,2016-07-24,2016-07-28,"GBC Prepunched Paper, 19-Hole, for Binding Sys...",Standard Class,90.060
350,CA-2017-129714,OFF-PA-10001970,2017-01-09,2017-03-09,Xerox 1881,First Class,24.560
352,CA-2017-129714,OFF-PA-10001970,2017-01-09,2017-03-09,Xerox 1881,First Class,49.120
1300,CA-2017-137043,FUR-FU-10003664,2017-12-23,2017-12-25,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",Second Class,572.760
1301,CA-2017-137043,FUR-FU-10003664,2017-12-23,2017-12-25,"Electrix Architect's Clamp-On Swing Arm Lamp, ...",Second Class,286.380
9168,CA-2017-140571,OFF-PA-10001954,2017-03-15,2017-03-19,Xerox 1964,Standard Class,319.760
9169,CA-2017-140571,OFF-PA-10001954,2017-03-15,2017-03-19,Xerox 1964,Standard Class,45.680
7881,CA-2018-118017,TEC-AC-10002006,2018-03-12,2018-06-12,Memorex Micro Travel Drive 16 GB,Second Class,76.752
7882,CA-2018-118017,TEC-AC-10002006,2018-03-12,2018-06-12,Memorex Micro Travel Drive 16 GB,Second Class,102.336


##### Code to safely clean the inaccurate entries
##### Steps
* check the shape before cleaning
* drop rows only where the 'Order Id', 'Product Id', and 'Sales' values are identical while keeping the distinct sales rows
* check the shape after cleaning
 
```
print(f"Shape before cleaning: {df.shape}")
df_cleaned = df.drop_duplicates(subset=['Order Id', 'Product Id', 'Sales'], keep='first')
print(f"Shape after cleaning: {df_cleaned.shape})
```

In [12]:
missing_order_dates = df['Order Date'].isna().sum()
print(f'Total rows with missing Order Date: {missing_order_dates}')

Total rows with missing Order Date: 0


In [13]:
missing_ship_dates = df['Ship Date'].isna().sum()
print(f'Total rows with missing Ship Date: {missing_ship_dates}')

Total rows with missing Ship Date: 0


In [14]:
missing_df = df[df['Order Date'].isna()]
display(missing_df[['Order Date', 'Ship Date', 'Customer Name', 'Product Name', 'Sales']].head())

,Order Date,Ship Date,Customer Name,Product Name,Sales


#####
Statistical summary on the sales column

In [15]:
df['Sales'].describe()

count     9800.000000
mean       230.769059
std        626.651875
min          0.444000
25%         17.248000
50%         54.490000
75%        210.605000
max      22638.480000
Name: Sales, dtype: float64

In [16]:
print(df['Sales'].describe())

count     9800.000000
mean       230.769059
std        626.651875
min          0.444000
25%         17.248000
50%         54.490000
75%        210.605000
max      22638.480000
Name: Sales, dtype: float64


In [17]:
df['Sales'].dtype

dtype('float64')

In [18]:
invalid_sales = df[pd.to_numeric(df['Sales'], errors='coerce').isna()]
print(f'Total rows with invalid Sales values: {len(invalid_sales)}')

Total rows with invalid Sales values: 0


##### Verifying Data Uniformity

In [19]:
categorical_columns = ['Ship Mode', 'Segment', 'Region', 'Category', 'Sub-Category']
print("--- Categorical Columns ---\n")

for cols in categorical_columns:
    # get uniques values and their count
    unique_values = df[cols].unique()
    unique_count = df[cols].nunique()

    print(f"Column: {cols} ({unique_count} unique values)")
    print(f"Unique Values: {unique_values}")
    print("-"*100)

--- Categorical Columns ---

Column: Ship Mode (4 unique values)
Unique Values: ['Second Class' 'Standard Class' 'First Class' 'Same Day']
----------------------------------------------------------------------------------------------------
Column: Segment (3 unique values)
Unique Values: ['Consumer' 'Corporate' 'Home Office']
----------------------------------------------------------------------------------------------------
Column: Region (4 unique values)
Unique Values: ['South' 'West' 'Central' 'East']
----------------------------------------------------------------------------------------------------
Column: Category (3 unique values)
Unique Values: ['Furniture' 'Office Supplies' 'Technology']
----------------------------------------------------------------------------------------------------
Column: Sub-Category (17 unique values)
Unique Values: ['Bookcases' 'Chairs' 'Labels' 'Tables' 'Storage' 'Furnishings' 'Art'
 'Phones' 'Binders' 'Appliances' 'Paper' 'Accessories' 'Envelopes'


#### Confirmed absolute data uniformity across the text columns